# 03 — Сопоставление аудио ↔ текст-шаблон

> **Статус прежних результатов:** `outputs/besy/run_01/matching_experiments/` устарели. Они были получены на прежнем неполном корпусе текста (1 086 475 символов) и не совместимы с текущим полным корпусом. Не используйте их позиции как anchors или метрики текущего эксперимента.
## Цель
Привязать аудиосегменты к позициям в тексте-шаблоне через взвешенную комбинацию трёх сигналов:

```
combined = α * char_tfidf + β * word_tfidf + γ * semantic
```

## Вход
- `normalized_text.pkl` — текущий полный нормализованный корпус
- `audio_segments.pkl` (35 260 сегментов)

## Выход
- `audio_map.pkl` — сопоставленные группы
- `unmatched.pkl` — несопоставленные группы
- `matching_experiments/` — артефакты для анализа

In [1]:
import pickle, json, os
from pathlib import Path
from datetime import datetime
import numpy as np

PROJECT_ROOT = Path(os.environ.get(
    "SPARK_ROOT",
    "/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit"
))
RUN_DIR = PROJECT_ROOT / "outputs/besy/run_01"
EXP_DIR = RUN_DIR / "matching_experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

def to_json_safe(obj):
    """Рекурсивно преобразовать numpy-типы в нативные для JSON."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    return obj

print("Базовые импорты готовы")

Базовые импорты готовы


In [2]:
# Загружаем входные данные
with open(RUN_DIR / "normalized_text.pkl", "rb") as f:
    text_data = pickle.load(f)
normalized_text = text_data["normalized_text"]
section_boundaries = text_data["section_boundaries"]

with open(RUN_DIR / "audio_segments.pkl", "rb") as f:
    audio_data = pickle.load(f)
audio_segments = audio_data["audio_segments"]
timeline = audio_data["timeline"]
total_duration = audio_data["total_duration"]

print(f"Текст: {len(normalized_text):,} символов")
print(f"Аудио: {len(audio_segments):,} сегментов, {total_duration/3600:.1f}ч")

Текст: 1,086,475 символов
Аудио: 35,260 сегментов, 36.2ч


## Шаг 1. Склейка сегментов в аудиогруппы

In [3]:
def build_audio_groups(segments, group_size_sec=30, min_group_sec=10):
    """Склеить сегменты в группы примерно по group_size_sec секунд.
    Не разрывает группу на границе файла, если остаток < min_group_sec.
    """
    groups = []
    buf_text, buf_start, buf_end, buf_file, buf_idx = [], None, None, None, None
    
    for i, s in enumerate(segments):
        if buf_start is None:
            buf_start = s["global_start"]
            buf_file = s["file"]
            buf_idx = s["file_index"]
        
        buf_text.append(s["text"])
        buf_end = s["global_end"]
        dur = buf_end - buf_start
        
        next_idx = i + 1 if i + 1 < len(segments) else None
        file_boundary = (next_idx is None) or (segments[next_idx]["file_index"] != buf_idx)
        
        if file_boundary and dur < min_group_sec:
            continue  # остаток → склеим со следующим файлом
        
        if dur >= group_size_sec or file_boundary:
            groups.append({
                "audio_start": buf_start, "audio_end": buf_end,
                "duration": dur, "text": " ".join(buf_text),
                "file": buf_file, "file_index": buf_idx,
                "segment_count": len(buf_text),
            })
            buf_text, buf_start = [], None
    
    if buf_text:
        groups.append({
            "audio_start": buf_start, "audio_end": buf_end,
            "duration": buf_end - buf_start, "text": " ".join(buf_text),
            "file": buf_file, "file_index": buf_idx,
            "segment_count": len(buf_text),
        })
    
    return groups

# Тест
for gs in [15, 30, 60]:
    g = build_audio_groups(audio_segments, group_size_sec=gs)
    print(f"group={gs:2d}с → {len(g):>5} групп, среднее={np.mean([x['duration'] for x in g]):.0f}с, {np.mean([len(x['text'].split()) for x in g]):.0f} слов")

group=15с →  7361 групп, среднее=17с, 29 слов
group=30с →  3999 групп, среднее=32с, 54 слов
group=60с →  2088 групп, среднее=62с, 104 слов


## Шаг 2. Нарезка текста на окна

In [4]:
def build_text_windows(text, window_size=400, step=100):
    """Нарезать текст на перекрывающиеся окна."""
    windows = []
    for start in range(0, len(text) - window_size, step):
        windows.append({
            "char_start": start,
            "char_end": start + window_size,
            "text": text[start:start + window_size],
        })
    # Последнее окно — до конца текста
    if windows and windows[-1]["char_end"] < len(text):
        windows.append({
            "char_start": len(text) - window_size,
            "char_end": len(text),
            "text": text[-window_size:],
        })
    return windows

for ws in [200, 400, 700]:
    w = build_text_windows(normalized_text, window_size=ws, step=100)
    print(f"window={ws:3d} → {len(w):>5} окон, среднее слов={np.mean([len(x['text'].split()) for x in w]):.0f}")

window=200 → 10864 окон, среднее слов=33
window=400 → 10862 окон, среднее слов=65
window=700 → 10859 окон, среднее слов=113


## Шаг 3. Функции matching

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def char_tfidf_match(group_texts, window_texts, top_k=5):
    """Символьный TF-IDF → top_k кандидатов для каждой группы."""
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=50000)
    all_texts = window_texts + group_texts
    tfidf = vec.fit_transform(all_texts)
    W = tfidf[:len(window_texts)]
    G = tfidf[len(window_texts):]
    results = []
    for i in range(G.shape[0]):
        sims = cosine_similarity(G[i], W)[0]
        top = np.argpartition(sims, -top_k)[-top_k:]
        top = top[np.argsort(sims[top])[::-1]]
        results.append([(int(t), float(sims[t])) for t in top])
    return results

def word_tfidf_match(group_texts, window_texts, top_k=5):
    """Словный TF-IDF → top_k кандидатов для каждой группы."""
    vec = TfidfVectorizer(analyzer='word', max_features=50000)
    all_texts = window_texts + group_texts
    tfidf = vec.fit_transform(all_texts)
    W = tfidf[:len(window_texts)]
    G = tfidf[len(window_texts):]
    results = []
    for i in range(G.shape[0]):
        sims = cosine_similarity(G[i], W)[0]
        top = np.argpartition(sims, -top_k)[-top_k:]
        top = top[np.argsort(sims[top])[::-1]]
        results.append([(int(t), float(sims[t])) for t in top])
    return results

print("TF-IDF функции готовы")

TF-IDF функции готовы


In [6]:
from sentence_transformers import SentenceTransformer

def semantic_score(model, group_text, candidate_texts):
    """Embedding косинусное сходство: группа vs кандидаты."""
    gv = model.encode([group_text], convert_to_numpy=True)
    cv = model.encode(candidate_texts, convert_to_numpy=True)
    return [float(s) for s in cosine_similarity(gv, cv)[0]]

def resolve_section(char_pos, boundaries):
    """Определить главу по позиции в тексте."""
    for start, end, title in boundaries:
        if start <= char_pos <= end:
            return title
    return "?"

print("Вспомогательные функции готовы")

Вспомогательные функции готовы


In [7]:
def compute_diagnostics(audio_map, unmatched, audio_groups):
    """Метрики качества matching."""
    n_total = len(audio_groups)
    n_matched = len(audio_map)
    n_unmatched = len(unmatched)
    
    d = {
        "total_groups": n_total,
        "matched": n_matched,
        "unmatched": n_unmatched,
        "unmatched_pct": round(100 * n_unmatched / n_total, 1),
    }
    
    if n_matched > 1:
        pos = [m["char_start"] for m in audio_map]
        backward = sum(1 for i in range(1, len(pos)) if pos[i] < pos[i-1])
        d["monotonicity_violations"] = backward
        d["monotonicity_violation_pct"] = round(100 * backward / (len(pos) - 1), 1)
        sizes = [pos[i-1] - pos[i] for i in range(1, len(pos)) if pos[i] < pos[i-1]]
        d["backward_mean_chars"] = round(float(np.mean(sizes)), 0) if sizes else 0
        d["backward_max_chars"] = round(float(np.max(sizes)), 0) if sizes else 0
    
    if n_matched > 0:
        boundaries = []
        for i in range(1, len(audio_map)):
            if audio_map[i]["file_index"] != audio_map[i-1]["file_index"]:
                gap = audio_map[i]["char_start"] - audio_map[i-1]["char_end"]
                boundaries.append(gap)
        d["file_boundaries"] = len(boundaries)
        d["boundary_gaps_mean"] = round(float(np.mean(np.abs(boundaries))), 0) if boundaries else 0
        d["boundary_gaps_max"] = round(float(np.max(np.abs(boundaries))), 0) if boundaries else 0
        
        scores = [m["combined"] for m in audio_map]
        d["combined_mean"] = round(float(np.mean(scores)), 4)
        d["combined_median"] = round(float(np.median(scores)), 4)
        d["combined_std"] = round(float(np.std(scores)), 4)
        for sig in ["char_tfidf", "word_tfidf", "semantic"]:
            sv = [m[sig] for m in audio_map]
            d[f"{sig}_mean"] = round(float(np.mean(sv)), 4)
    
    if n_unmatched > 0:
        uc = [u["best_combined"] for u in unmatched]
        d["unmatched_combined_max"] = round(float(np.max(uc)), 4)
        d["unmatched_combined_mean"] = round(float(np.mean(uc)), 4)
    
    return d

In [8]:
def run_matching_pipeline(audio_groups, text_windows, alpha, beta, gamma,
                          tfidf_top_k=5, unmatched_threshold=0.15):
    """Полный пайплайн matching."""
    group_texts = [g["text"] for g in audio_groups]
    window_texts = [w["text"] for w in text_windows]
    
    print(f"  Char TF-IDF ({len(audio_groups)}×{len(text_windows)})…", end=" ", flush=True)
    char_res = char_tfidf_match(group_texts, window_texts, top_k=tfidf_top_k)
    print("OK")
    
    print(f"  Word TF-IDF…", end=" ", flush=True)
    word_res = word_tfidf_match(group_texts, window_texts, top_k=tfidf_top_k)
    print("OK")
    
    print(f"  Merge → top-{tfidf_top_k}…", end=" ", flush=True)
    merged = []
    for i in range(len(audio_groups)):
        cd = {}
        for idx, sc in char_res[i]:
            cd[idx] = {"char": sc, "word": 0.0}
        for idx, sc in word_res[i]:
            cd[idx] = {"char": cd.get(idx, {}).get("char", 0.0), "word": sc}
        cands = sorted(cd.items(), key=lambda x: (x[1]["char"] + x[1]["word"]) / 2, reverse=True)
        merged.append(cands[:tfidf_top_k])
    print("OK")
    
    print(f"  Semantic scoring…", end=" ", flush=True)
    sem_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    sem_scores = []
    for i, g in enumerate(audio_groups):
        idxs = [idx for idx, _ in merged[i]]
        txts = [text_windows[idx]["text"] for idx in idxs]
        sem_scores.append(semantic_score(sem_model, g["text"], txts))
    print("OK")
    
    print(f"  Weighted combination (α={alpha} β={beta} γ={gamma} thresh={unmatched_threshold})…", end=" ", flush=True)
    audio_map, unmatched = [], []
    for i, g in enumerate(audio_groups):
        best_c, best = -1, None
        for j, (wi, sc) in enumerate(merged[i]):
            combined = alpha * sc["char"] + beta * sc["word"] + gamma * sem_scores[i][j]
            if combined > best_c:
                best_c = combined
                best = (wi, sc["char"], sc["word"], sem_scores[i][j], combined)
        
        if best_c < unmatched_threshold:
            unmatched.append({
                "audio_start": g["audio_start"], "audio_end": g["audio_end"],
                "duration": g["duration"], "text": g["text"][:200],
                "file": g["file"], "best_combined": round(best_c, 4),
            })
        else:
            wi, ch, wo, se, co = best
            audio_map.append({
                "audio_start": g["audio_start"], "audio_end": g["audio_end"],
                "char_start": text_windows[wi]["char_start"],
                "char_end": text_windows[wi]["char_end"],
                "file": g["file"], "file_index": g["file_index"],
                "section": resolve_section(text_windows[wi]["char_start"], section_boundaries),
                "char_tfidf": round(ch, 4), "word_tfidf": round(wo, 4),
                "semantic": round(se, 4), "combined": round(co, 4),
                "audio_text": g["text"][:200],
                "text_snippet": normalized_text[text_windows[wi]["char_start"]:text_windows[wi]["char_start"]+200],
            })
    print("OK")
    
    diag = compute_diagnostics(audio_map, unmatched, audio_groups)
    return audio_map, unmatched, diag

## Эксперимент 1: базовый прогон

In [9]:
GROUP_SIZE = 30
WINDOW_SIZE = 400
WINDOW_STEP = 100
ALPHA = 0.4
BETA  = 0.3
GAMMA = 0.3
UNMATCHED_THRESHOLD = 0.15

audio_groups = build_audio_groups(audio_segments, group_size_sec=GROUP_SIZE)
text_windows = build_text_windows(normalized_text, window_size=WINDOW_SIZE, step=WINDOW_STEP)

print(f"Аудиогрупп: {len(audio_groups)}, окон: {len(text_windows)}")

audio_map, unmatched, diagnostics = run_matching_pipeline(
    audio_groups, text_windows, ALPHA, BETA, GAMMA,
    unmatched_threshold=UNMATCHED_THRESHOLD
)

print(f"\n=== ДИАГНОСТИКА ===")
for k, v in diagnostics.items():
    print(f"  {k}: {v}")

Аудиогрупп: 3999, окон: 10862
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK

=== ДИАГНОСТИКА ===
  total_groups: 3999
  matched: 3999
  unmatched: 0
  unmatched_pct: 0.0
  monotonicity_violations: 315
  monotonicity_violation_pct: 7.9
  backward_mean_chars: 279927.0
  backward_max_chars: 1001700.0
  file_boundaries: 23
  boundary_gaps_mean: 30926.0
  boundary_gaps_max: 518900.0
  combined_mean: 0.7365
  combined_median: 0.8264
  combined_std: 0.2114
  char_tfidf_mean: 0.7079
  word_tfidf_mean: 0.6807
  semantic_mean: 0.8305


In [10]:
# Сохраняем артефакты базового прогона
base_dir = EXP_DIR / "baseline"
base_dir.mkdir(exist_ok=True)

config = {
    "group_size": GROUP_SIZE, "window_size": WINDOW_SIZE,
    "window_step": WINDOW_STEP, "alpha": ALPHA, "beta": BETA, "gamma": GAMMA,
    "unmatched_threshold": UNMATCHED_THRESHOLD,
    "timestamp": datetime.now().isoformat(),
}

with open(base_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)
with open(base_dir / "diagnostics.json", "w") as f:
    json.dump(to_json_safe(diagnostics), f, indent=2)
with open(base_dir / "audio_map.pkl", "wb") as f:
    pickle.dump(audio_map, f)
with open(base_dir / "unmatched.pkl", "wb") as f:
    pickle.dump(unmatched, f)

# 15 точек для ручной проверки
with open(base_dir / "manual_check_samples.txt", "w", encoding="utf-8") as f:
    f.write("=== 15 ТОЧЕК ДЛЯ РУЧНОЙ ПРОВЕРКИ ===\n\n")
    rng = np.random.RandomState(42)
    indices = sorted(rng.choice(len(audio_map), min(15, len(audio_map)), replace=False))
    for idx in indices:
        m = audio_map[idx]
        mm, ss = int(m["audio_start"] // 60), int(m["audio_start"] % 60)
        f.write(f"[{mm:3d}:{ss:02d}] {m['file']}\n")
        f.write(f"  Аудио: \"{m['audio_text'][:200]}\"\n")
        f.write(f"  Текст: \"{m['text_snippet'][:200]}\"\n")
        f.write(f"  Глава: {m['section']}\n")
        f.write(f"  Scores: char={m['char_tfidf']:.3f} word={m['word_tfidf']:.3f} sem={m['semantic']:.3f} comb={m['combined']:.3f}\n\n")
    f.write("\n=== НЕСОПОСТАВЛЕННЫЕ ===\n\n")
    for u in unmatched[:30]:
        mm, ss = int(u["audio_start"] // 60), int(u["audio_start"] % 60)
        f.write(f"[{mm:3d}:{ss:02d}] comb={u['best_combined']:.3f} | \"{u['text'][:150]}\"\n")

print(f"Сохранено в {base_dir}/")

Сохранено в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/matching_experiments/baseline/


## Эксперимент 2: сетка параметров

In [11]:
def run_experiment(name, group_size, window_size, window_step, alpha, beta, gamma, threshold):
    """Запустить один эксперимент и сохранить config + diagnostics."""
    groups = build_audio_groups(audio_segments, group_size_sec=group_size)
    windows = build_text_windows(normalized_text, window_size=window_size, step=window_step)
    am, um, diag = run_matching_pipeline(groups, windows, alpha, beta, gamma, unmatched_threshold=threshold)
    
    exp_dir = EXP_DIR / name
    exp_dir.mkdir(exist_ok=True)
    
    config = {
        "group_size": group_size, "window_size": window_size,
        "window_step": window_step, "alpha": alpha, "beta": beta, "gamma": gamma,
        "unmatched_threshold": threshold,
        "n_groups": len(groups), "n_windows": len(windows),
    }
    with open(exp_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    with open(exp_dir / "diagnostics.json", "w") as f:
        json.dump(to_json_safe(diag), f, indent=2)
    return diag

print("Функция run_experiment готова")

Функция run_experiment готова


In [12]:
# 2.1. Размер аудиогруппы
print("=== Группа ===")
group_results = []
for gs in [15, 30, 60]:
    d = run_experiment(f"group_{gs}s", gs, 400, 100, 0.4, 0.3, 0.3, 0.15)
    group_results.append({"group_size": gs, **to_json_safe(d)})
    print(f"  {gs:2d}с: matched={d['matched']}, unmatched={d['unmatched_pct']}%, monot={d['monotonicity_violation_pct']}%")

with open(EXP_DIR / "group_size_sweep.json", "w") as f:
    json.dump(group_results, f, indent=2)

=== Группа ===
  Char TF-IDF (7361×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  15с: matched=7356, unmatched=0.1%, monot=10.4%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  30с: matched=3999, unmatched=0.0%, monot=7.9%
  Char TF-IDF (2088×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  60с: matched=2088, unmatched=0.0%, monot=6.3%


In [13]:
# 2.2. Размер окна
print("\n=== Окно ===")
window_results = []
for ws in [200, 400, 700]:
    d = run_experiment(f"window_{ws}", 30, ws, 100, 0.4, 0.3, 0.3, 0.15)
    window_results.append({"window_size": ws, **to_json_safe(d)})
    print(f"  {ws:3d}: matched={d['matched']}, unmatched={d['unmatched_pct']}%, monot={d['monotonicity_violation_pct']}%")

with open(EXP_DIR / "window_size_sweep.json", "w") as f:
    json.dump(window_results, f, indent=2)


=== Окно ===
  Char TF-IDF (3999×10864)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  200: matched=3999, unmatched=0.0%, monot=8.3%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  400: matched=3999, unmatched=0.0%, monot=7.9%
  Char TF-IDF (3999×10859)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.4 β=0.3 γ=0.3 thresh=0.15)… OK
  700: matched=3999, unmatched=0.0%, monot=8.9%


In [14]:
# 2.3. Веса α, β, γ (шаг 0.2, сумма = 1)
print("\n=== Веса ===")
combos = []
for a in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    for b in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
        g = round(1.0 - a - b, 1)
        if 0.0 <= g <= 1.0:
            combos.append((a, b, g))
combos = sorted(set(combos))
print(f"Комбинаций: {len(combos)}")

weight_results = []
for a, b, g in combos:
    name = f"weights_a{a:.1f}_b{b:.1f}_g{g:.1f}"
    d = run_experiment(name, 30, 400, 100, a, b, g, 0.15)
    weight_results.append({"alpha": a, "beta": b, "gamma": g, **to_json_safe(d)})

weight_results.sort(key=lambda x: (x['monotonicity_violation_pct'], -x['matched']))
print("\nТоп-5:")
for r in weight_results[:5]:
    print(f"  α={r['alpha']:.1f} β={r['beta']:.1f} γ={r['gamma']:.1f} | matched={r['matched']} unmatched={r['unmatched_pct']}% monot={r['monotonicity_violation_pct']}%")

with open(EXP_DIR / "weight_sweep.json", "w") as f:
    json.dump(weight_results, f, indent=2)


=== Веса ===
Комбинаций: 21
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=0.0 γ=1.0 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=0.2 γ=0.8 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=0.4 γ=0.6 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=0.6 γ=0.4 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=0.8 γ=0.2 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.15)… OK
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5…

In [15]:
# 2.4. Порог unmatched (на лучших весах)
best_w = weight_results[0]
print(f"\n=== Порог (α={best_w['alpha']:.1f} β={best_w['beta']:.1f} γ={best_w['gamma']:.1f}) ===")
threshold_results = []
for th in [0.05, 0.10, 0.15, 0.20, 0.25]:
    d = run_experiment(f"threshold_{int(th*100)}", 30, 400, 100, best_w['alpha'], best_w['beta'], best_w['gamma'], th)
    threshold_results.append({"threshold": th, **to_json_safe(d)})
    print(f"  {th:.2f}: matched={d['matched']}, unmatched={d['unmatched_pct']}%, monot={d['monotonicity_violation_pct']}%")

with open(EXP_DIR / "threshold_sweep.json", "w") as f:
    json.dump(threshold_results, f, indent=2)


=== Порог (α=0.0 β=1.0 γ=0.0) ===
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.05)… OK
  0.05: matched=3819, unmatched=4.5%, monot=6.0%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.1)… OK
  0.10: matched=3737, unmatched=6.6%, monot=4.8%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.15)… OK
  0.15: matched=3563, unmatched=10.9%, monot=2.1%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.2)… OK
  0.20: matched=3468, unmatched=13.3%, monot=0.6%
  Char TF-IDF (3999×10862)… OK
  Word TF-IDF… OK
  Merge → top-5… OK
  Semantic scoring… OK
  Weighted combination (α=0.0 β=1.0 γ=0.0 thresh=0.25)… OK
  0.25: matche

In [16]:
# 2.5. Сводка
print("\n" + "="*60)
print("СВОДКА")
print("="*60)

print("\nГруппа:")
for r in sorted(group_results, key=lambda x: x['monotonicity_violation_pct']):
    print(f"  {r['group_size']}с → monot={r['monotonicity_violation_pct']}%, matched={r['matched']}, unmatched={r['unmatched_pct']}%")

print("\nОкно:")
for r in sorted(window_results, key=lambda x: x['monotonicity_violation_pct']):
    print(f"  {r['window_size']}с → monot={r['monotonicity_violation_pct']}%, matched={r['matched']}")

print("\nВеса (топ-3):")
for r in weight_results[:3]:
    print(f"  α={r['alpha']:.1f} β={r['beta']:.1f} γ={r['gamma']:.1f} → monot={r['monotonicity_violation_pct']}%, comb_mean={r['combined_mean']}")

print("\nПорог:")
for r in sorted(threshold_results, key=lambda x: x['monotonicity_violation_pct']):
    print(f"  {r['threshold']:.2f} → monot={r['monotonicity_violation_pct']}%, matched={r['matched']}, unmatched={r['unmatched_pct']}%")

print(f"\nАртефакты в {EXP_DIR}/")


СВОДКА

Группа:
  60с → monot=6.3%, matched=2088, unmatched=0.0%
  30с → monot=7.9%, matched=3999, unmatched=0.0%
  15с → monot=10.4%, matched=7356, unmatched=0.1%

Окно:
  400с → monot=7.9%, matched=3999
  200с → monot=8.3%, matched=3999
  700с → monot=8.9%, matched=3999

Веса (топ-3):
  α=0.0 β=1.0 γ=0.0 → monot=2.1%, comb_mean=0.7606
  α=0.2 β=0.8 γ=0.0 → monot=2.3%, comb_mean=0.7588
  α=0.4 β=0.6 γ=0.0 → monot=3.2%, comb_mean=0.7539

Порог:
  0.25 → monot=0.3%, matched=3426, unmatched=14.3%
  0.20 → monot=0.6%, matched=3468, unmatched=13.3%
  0.15 → monot=2.1%, matched=3563, unmatched=10.9%
  0.10 → monot=4.8%, matched=3737, unmatched=6.6%
  0.05 → monot=6.0%, matched=3819, unmatched=4.5%

Артефакты в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/matching_experiments/


## Артефакты для анализа

```
outputs/besy/run_01/matching_experiments/
├── baseline/
│   ├── config.json               — параметры
│   ├── diagnostics.json          — метрики
│   ├── audio_map.pkl             — карта сопоставления
│   ├── unmatched.pkl             — несопоставленные
│   └── manual_check_samples.txt  — 15 точек для ручной проверки
├── group_15s/    — эксперименты с размером группы
├── group_30s/
├── group_60s/
├── window_200/   — эксперименты с размером окна
├── window_400/
├── window_700/
├── weights_*/    — эксперименты с весами
├── threshold_*/  — эксперименты с порогом
├── *_sweep.json   — сводки
```

## Критерии оценки

1. **Монотонность** — violations < 5% → отлично, 5–15% → приемлемо, >15% → проблемы
2. **Unmatched** — 5–15% → норма. <2% → порог мягкий, >25% → порог жёсткий
3. **Ручная проверка** → `manual_check_samples.txt`